# 01 — Data Generation & Cleaning
**Email Marketing Funnel Analysis**

This notebook simulates a realistic 10,000-row email-marketing dataset, runs data-quality checks (nulls, duplicates, types), enforces funnel logical consistency (a click can only happen after an open), and exports both raw and cleaned CSV files.

**Funnel logic:** `Sent → Delivered → Opened → Clicked → Converted`

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)   # reproducibility
N_ROWS = 10_000

## 1. Simulate categorical dimensions and send dates

In [2]:
n = N_ROWS
campaign_type = np.random.choice(['Welcome','Promotional','Newsletter'], n, p=[0.25,0.45,0.30])
subject_line  = np.random.choice(['Version_A','Version_B'], n, p=[0.5,0.5])
device_type   = np.random.choice(['Mobile','Desktop','Tablet'], n, p=[0.55,0.35,0.10])
region        = np.random.choice(['North','South','East','West'], n)
customer_segment = np.random.choice(['New','Returning','VIP'], n, p=[0.45,0.40,0.15])

end_date = pd.Timestamp('2025-06-15')
start_date = end_date - pd.Timedelta(days=182)   # ~6 months
sent_date = start_date + pd.to_timedelta(np.random.randint(0,183,n), unit='D')

df = pd.DataFrame({
    'email_id': [f'EM{100000+i}' for i in range(n)],
    'campaign_type': campaign_type, 'subject_line': subject_line,
    'sent_date': sent_date, 'device_type': device_type,
    'region': region, 'customer_segment': customer_segment,
})
df.head()

,email_id,campaign_type,subject_line,sent_date,device_type,region,customer_segment
0,EM100000,Promotional,Version_A,2025-01-25,Desktop,North,Returning
1,EM100001,Newsletter,Version_A,2025-06-12,Mobile,East,New
2,EM100002,Newsletter,Version_A,2025-06-06,Mobile,South,Returning
3,EM100003,Promotional,Version_B,2025-01-23,Desktop,North,New
4,EM100004,Welcome,Version_A,2025-01-07,Mobile,South,Returning


## 2. Simulate the funnel with realistic probabilities
- Delivery 95% · Open 28-35% · CTR 8-12% · Conversion 2-5% · Unsubscribe 1-2%
- `Version_B` subject lines get a small open-rate boost to create a detectable A/B effect.
- Desktop & VIP customers convert better; Mobile slightly worse (realistic).

In [3]:
# 1) Delivered
df['delivered'] = np.random.binomial(1, 0.95, n)

# 2) Opened (only if delivered)
base_open = np.where(df['subject_line']=='Version_B', 0.335, 0.295)
camp_mod = df['campaign_type'].map({'Welcome':0.04,'Promotional':-0.01,'Newsletter':0.0})
open_p = np.clip(base_open + camp_mod, 0, 1)
df['opened'] = np.where(df['delivered']==1, np.random.binomial(1, open_p), 0)

# 3) Clicked (only if opened)
ctor_p = np.clip(0.30 + df['campaign_type'].map({'Welcome':0.05,'Promotional':0.03,'Newsletter':-0.02}),0,1)
df['clicked'] = np.where(df['opened']==1, np.random.binomial(1, ctor_p), 0)

# 4) Converted (only if clicked)
conv_p = 0.30 + df['device_type'].map({'Desktop':0.06,'Tablet':0.0,'Mobile':-0.05})
conv_p = np.clip(conv_p + df['customer_segment'].map({'VIP':0.10,'Returning':0.03,'New':-0.04}),0,1)
df['converted'] = np.where(df['clicked']==1, np.random.binomial(1, conv_p), 0)

# 5) Unsubscribed (only if delivered)
df['unsubscribed'] = np.where(df['delivered']==1, np.random.binomial(1,0.015,n), 0)
df.to_csv('../data/raw_email_data.csv', index=False)
df.head()

,email_id,campaign_type,subject_line,sent_date,device_type,region,customer_segment,delivered,opened,clicked,converted,unsubscribed
0,EM100000,Promotional,Version_A,2025-01-25,Desktop,North,Returning,1,0,0,0,0
1,EM100001,Newsletter,Version_A,2025-06-12,Mobile,East,New,1,0,0,0,0
2,EM100002,Newsletter,Version_A,2025-06-06,Mobile,South,Returning,1,0,0,0,0
3,EM100003,Promotional,Version_B,2025-01-23,Desktop,North,New,0,0,0,0,0
4,EM100004,Welcome,Version_A,2025-01-07,Mobile,South,Returning,1,1,0,0,0


## 3. Data-quality checks

In [4]:
print('Shape:', df.shape)
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate email_id:', df['email_id'].duplicated().sum())
print('\nNulls:\n', df.isnull().sum())
print('\nLogical violations:')
print('  opened & not delivered :', ((df.opened==1)&(df.delivered==0)).sum())
print('  clicked & not opened   :', ((df.clicked==1)&(df.opened==0)).sum())
print('  converted & not clicked:', ((df.converted==1)&(df.clicked==0)).sum())

Shape: (10000, 12)
Duplicate rows: 0
Duplicate email_id: 0

Nulls:
 email_id            0
campaign_type       0
subject_line        0
sent_date           0
device_type         0
region              0
customer_segment    0
delivered           0
opened              0
clicked             0
converted           0
unsubscribed        0
dtype: int64

Logical violations:
  opened & not delivered : 0
  clicked & not opened   : 0
  converted & not clicked: 0


## 4. Clean: enforce consistency, fix types, add helper columns, export

In [5]:
df.loc[df.delivered==0, ['opened','clicked','converted','unsubscribed']] = 0
df.loc[df.opened==0, ['clicked','converted']] = 0
df.loc[df.clicked==0, 'converted'] = 0

df['sent_date'] = pd.to_datetime(df['sent_date'])
df['bounced'] = (df['delivered']==0).astype(int)
df['sent_week'] = df['sent_date'].dt.to_period('W').dt.start_time
df = df.drop_duplicates(subset='email_id').reset_index(drop=True)

df.to_csv('../data/cleaned_email_data.csv', index=False)
print('Cleaned data exported. Rows:', len(df))
df.head()

Cleaned data exported. Rows: 10000


,email_id,campaign_type,subject_line,sent_date,device_type,region,customer_segment,delivered,opened,clicked,converted,unsubscribed,bounced,sent_week
0,EM100000,Promotional,Version_A,2025-01-25,Desktop,North,Returning,1,0,0,0,0,0,2025-01-20
1,EM100001,Newsletter,Version_A,2025-06-12,Mobile,East,New,1,0,0,0,0,0,2025-06-09
2,EM100002,Newsletter,Version_A,2025-06-06,Mobile,South,Returning,1,0,0,0,0,0,2025-06-02
3,EM100003,Promotional,Version_B,2025-01-23,Desktop,North,New,0,0,0,0,0,1,2025-01-20
4,EM100004,Welcome,Version_A,2025-01-07,Mobile,South,Returning,1,1,0,0,0,0,2025-01-06


✅ **Output:** `data/raw_email_data.csv` and `data/cleaned_email_data.csv`.

Continue to **02_funnel_analysis.ipynb**.